# Data Preprocessing: Craigslist Used Cars

Raw data: 426,880 postings × 26 columns. Goal: a clean table ready for analysis. Thresholds are set once in the first code cell.

## 1. Load data

Limits chosen from the raw-data profile:
- **Price:** $500–$150,000 (32,895 prices are 0, about 9,000 more are $1–$499, and the max is $3.7 billion).
- **Year:** 1950–2022 (1900 is a placeholder; 2022 is valid for 2021 listings).
- **Odometer:** up to 500,000 miles (99th percentile is 280k; the max is 10,000,000).

In [ ]:
import glob
import numpy as np
import pandas as pd

PRICE_MIN, PRICE_MAX = 500, 150_000
YEAR_MIN, YEAR_MAX = 1950, 2022
ODO_MAX = 500_000

path = (glob.glob("/kaggle/input/**/vehicles.csv", recursive=True) or glob.glob("vehicles.csv"))[0]
df = pd.read_csv(path)
raw_rows = len(df)
print("Shape:", df.shape)

## 2. Duplicates

Each row is a posting identified by `id`, so duplicates are checked on `id` only. `id` is unique and there are no full-row duplicates, so **nothing is dropped**.

`drop_duplicates()` is not used after the identifier columns are removed, because that would delete different postings that just share the same attributes.

In [ ]:
print("Repeated ids:", df["id"].duplicated().sum())
print("Full-row duplicates:", df.duplicated().sum())

## 3. Drop unused columns and clean text

- Dropped: identifiers and links (`id`, `url`, `region_url`, `image_url`, `description`, `VIN`), `county` (100% missing), `size` (72% missing), and `lat`/`long` (`state` and `region` already give location).
- Text is stripped and lowercased (`model` has about 1,091 case-only variants).
- `posting_date` is converted to datetime.

In [ ]:
df = df.drop(columns=["id", "url", "region_url", "image_url", "description",
                      "VIN", "county", "size", "lat", "long"])

text_cols = ["region", "manufacturer", "model", "condition", "cylinders", "fuel",
             "title_status", "transmission", "drive", "type", "paint_color", "state"]
for c in text_cols:
    df[c] = df[c].astype("object").str.strip().str.lower().replace("", np.nan)

df["posting_date"] = pd.to_datetime(df["posting_date"], errors="coerce", utc=True)

## 4. Invalid values

Domain limits are used instead of IQR, because price and odometer are very right-skewed and IQR would remove thousands of normal listings.

Placeholder numbers such as `11111`, `123456` and `9999999` are also removed. An odometer of 0 is kept only for new cars.

In [ ]:
def is_placeholder(x):
    if pd.isna(x):
        return False
    s = str(int(x))
    return len(s) >= 5 and (len(set(s)) == 1
                            or s in "01234567890123456789"
                            or s in "98765432109876543210")

is_new = (df["condition"] == "new") | (df["year"] >= 2021)

bad = (
    (df["price"] < PRICE_MIN) | (df["price"] > PRICE_MAX) | df["price"].map(is_placeholder)
    | (df["year"].notna() & ~df["year"].between(YEAR_MIN, YEAR_MAX))
    | (df["odometer"] > ODO_MAX) | df["odometer"].map(is_placeholder)
    | ((df["odometer"] == 0) & ~is_new)
)
print("Invalid rows removed:", bad.sum())
df = df[~bad].copy()

## 5. Missing values

- **Nearly-empty rows** (fewer than 6 of 12 core fields filled) are dropped. Filling them would invent a vehicle.
- **Missing `year` or `odometer`** (about 0.3% and 1% of rows) are dropped instead of median-filled.
- **Missing categories** become `"unknown"`, which keeps the rows.

In [ ]:
core = ["year", "odometer", "manufacturer", "model", "condition", "cylinders",
        "fuel", "title_status", "transmission", "drive", "type", "paint_color"]

df = df[df[core].notna().sum(axis=1) >= 6]
df = df.dropna(subset=["year", "odometer"]).copy()
df["year"] = df["year"].astype(int)
df[text_cols] = df[text_cols].fillna("unknown")

## 6. Final check

Report only, no changes.

In [ ]:
print(f"Final shape: {df.shape}  (kept {len(df)/raw_rows:.1%} of raw rows)")
print("Missing values:", df.isna().sum().sum())
df[["price", "year", "odometer"]].describe().round(0)